In [1]:
%load_ext autoreload
%autoreload 2
%reload_ext autoreload

import nest_asyncio
nest_asyncio.apply()

import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = "vscode"            

import matplotlib.pyplot as plt
import matplotlib.pylab as pylab
import matplotlib.dates as mdates
plt.style.use('ggplot')
params = {'legend.fontsize': 'x-large',
        'figure.figsize': (12, 8),
        'axes.labelsize': 'x-large',
        'axes.titlesize':'x-large',
        'xtick.labelsize':'x-large',
        'ytick.labelsize':'x-large'}
pylab.rcParams.update(params)

import pandas as pd
import numpy as np

import datetime
import pytz
NY_tz = pytz.timezone("America/New_York") 
CHI_tz = pytz.timezone("America/Chicago") 
UTC_tz = pytz.timezone("UTC") 

In [20]:
import rateslib as rl
import QuantLib as ql

from MDP.IRSwaps.IRSwapsMDP import IRSwapsMDP
from Query.IRSwaps.IRSwapQuery import IRSwapQuery, IRSwapStructure
from Query.IRSwaps.IRSwapStructure import IRSwapStructureFunctionMap
from Query.IRSwaps.IRSwapValue import IRSwapValue, IRSwapValueFunctionMap

# fmt: off
import Query.IRSwaps.adapter  # noqa: F401
# fmt: on

from utils.ql_utils import datetime_to_ql_date, ql_date_to_datetime 

# Fetch Curve

In [21]:
# curve_mdp = IRSwapsMDP(source="SDR_INTRADAY-rl_usd_sofr_mtv2_q12x11")
curve_mdp = IRSwapsMDP(source="CME_NY_EOD_LIVE-ql_basic")

In [22]:
curve = "USD-SOFR-1D"
timestamp = datetime.date(2025, 9, 30)

curve_handle = curve_mdp._get_curve(curve_name=curve, timestamp=timestamp)

## Price Outright by tenor

In [28]:
risk = 100_000
outright_query = IRSwapQuery(curve=curve, tenor="10Y", structure=IRSwapStructure.OUTRIGHT, structure_kwargs={"bpv": risk})
outright_pkg, outright_rws = outright_query.resolve_package(pricer_or_curve=curve_handle) 

outright_vmap = outright_query.build_value_map(pricer_or_curve=curve_handle, package=outright_pkg, risk_weights=outright_rws)
for v in [IRSwapValue.RATE, IRSwapValue.NPV, IRSwapValue.NOTIONAL, IRSwapValue.PV01, IRSwapValue.DV01]:
    print(v.name, outright_vmap.apply(value=v))

RATE 3.6616523458076333
NPV 7.450580596923828e-09
NOTIONAL 119029120.07501258
PV01 100000.0
DV01 102851.48082371056


In [24]:
# irsvfp.apply(value=IRSwapValue.CARRY_BPS_RUNNING, **{"horizon": "1m"})
# outright_vmap.apply(value=IRSwapValue.CARRY_AND_ROLL_BPS_RUNNING, **{"horizon": "9m"})

# Price Curve

In [38]:
risk = -100_000
curve_query = IRSwapQuery(curve=curve, structure=IRSwapStructure.CURVE, structure_kwargs={"front_tenor": "2Y", "back_tenor": "10Y", "bpv": risk})
curve_pkg, curve_rws = curve_query.resolve_package(pricer_or_curve=curve_handle)

curve_vmap = curve_query.build_value_map(pricer_or_curve=curve_handle, package=curve_pkg, risk_weights=curve_rws)
for v in [IRSwapValue.RATE, IRSwapValue.NPV, IRSwapValue.PV01, IRSwapValue.DV01]:
    print(v.name, curve_vmap.apply(value=v))

RATE 27.324859554040334
NPV -3.725290298461914e-08
PV01 0.0
DV01 708.7816271521151
